# send the ground truth to gemini

In [6]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

# === Load API Key ===
load_dotenv()

model_name = "gemini-2.5-pro"
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))
model = genai.GenerativeModel(model_name)

def send_pdf_to_gemini_and_save_json(pdf_path, prompt, output_base_dir):
    try:
        # Upload the PDF file
        file_resource = genai.upload_file(pdf_path, mime_type="application/pdf")
        
        # Compose the prompt and file
        response = model.generate_content([prompt, file_resource])
        generated_text = response.text

        # Extract the filename and set output directory for JSON file
        pdf_filename = os.path.basename(pdf_path)
        pdf_stem = os.path.splitext(pdf_filename)[0]
        output_dir = os.path.join(output_base_dir, pdf_stem)
        os.makedirs(output_dir, exist_ok=True)

        # Define the output JSON path
        output_json_path = os.path.join(output_dir, f"{pdf_stem}.json")

        # Write the output JSON file
        with open(output_json_path, 'w', encoding='utf-8') as output_file:
            output_file.write(generated_text)
        
        print(f"🎉 Output saved to {output_json_path}")
    except Exception as e:
        print(f"❌ Error in generating response for {pdf_path}: {str(e)}")

# Function to process all PDF files in a directory
def process_all_pdfs(input_dir, output_dir, prompt):
    # Iterate through all files in the directory
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".pdf"):
                input_pdf_path = os.path.join(root, file)
                print(f"Processing {input_pdf_path}...")
                send_pdf_to_gemini_and_save_json(input_pdf_path, prompt, output_dir)

# === Example Usage ===
if __name__ == "__main__":
    # Input directory containing PDF files
    input_pdf_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/Physics_pdf_docx_human_ocr"  # Change this to the correct directory path
    output_json_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/pdf_docx_json"  # Change this to the desired output directory
    prompt = """
**Role**: You are a meticulous digital archivist tasked with transcribing PDF.

**Core Task**: Your goal is to create a perfect digital copy PDF. You must transcribe the text *exactly* as it appears, including any spelling or grammatical errors.

---

### Input Format

*Schema Definitions:**
- `question_number` (integer): The main question number.
- `ocr_text` (string): The full, collated text for the question and all its sub-parts.
- `diagrams` (array): A list of diagram objects. Leave as an empty array `[]` if none.
  - `id` (string): The diagram identifier from the text.
  - `coordinates` (string): "x_mid,y_mid,width,height", with values normalized between 0 and 1 relative to image dimensions.
  - `page_number` (integer): The page where the diagram is located.
- `pages` (array): A list of all page numbers on which any part of the question appears.

---

### Other Directives
1.  **Collate Sub-Questions**: Group all parts of a question (e.g., 11. (1), 11. (2)) under a single main question number. Preserve the original sub-question numbering in the text.
2.  **Angle Bracket Content**: Do NOT transcribe any text or information that is enclosed within angle brackets (`<TEXT>`). These indicate placeholders for diagrams or metadata that should not be part of the `ocr_text`. Instead, use the information within angle brackets to identify and populate the `diagrams` array, matching `suggested_id` or extracting the ID directly from the text within the brackets (e.g., `<diagram_1>` means the ID is `diagram_1`).
3.  **Diagram Matching**: When a diagram ID is mentioned in the `ocr_text` (e.g., "See <figure_A>"), ensure that the corresponding diagram in the `diagrams` array uses that exact ID and is correctly associated with the `ocr_text` for that question. If a diagram exists in `detected_diagrams` but is *not* referenced in the `ocr_text` with an `<ID>` tag, still include it in the `diagrams` array for the relevant question, using its `id` or an auto-generated one if `id` is missing.
4.  **Schema Adherence**: Strictly adhere to the output JSON schema and data types provided below.
5.  **Mathematical Notation**: Normalize mathematical expressions by converting vertical fractions (e.g., `A\nB`) to `A/B`, interpreting potential exponents (e.g., `X-Y` as `X^Y` if `Y` is a number or common variable for power), and standardizing subscripts (e.g., `A₁` or `A1` where context implies to `A_1`).
6. Your output must visually replicate all elements, including the diagonal strikethrough on the fraction. Use appropriate formatting (like LaTeX or Unicode) to show the strikethrough
7. **Table Formatting** : Extract the text from the table, including the column names and rows. Ensure the table structure is preserved exactly as it appears, and format it into a well-structured text format.Convert the following table into a structured format. If any cell contains an image or diagram, represent it as <diagram_X>, where X is the diagram identifier (e.g., <diagram_1>, <diagram_2>, etc.).
---

### Output Format
- The output MUST be a single, valid JSON  containing one object per main question.
- Do NOT include any text or explanations outside of the JSON .

Example Input for your processing (you would replace this with your actual data):
[
 {
    "question_number": 11,
    "ocr_text": "11. (1) This is the answer to the first part. 11. (2) This is the final answer, which contains a PV curve <diagram_1>. [NOTE: The student originally wrote 'the initial answer' here and struck it through; it has been correctly omitted from this output per the critical rule.]<diagram_1>",
    "diagrams": [
      {
        "id": "diagram_1",
        "coordinates": "0.5,0.5,0.2,0.3",
        "diagram_class": "diagram",
        "page_number": 3
      }
    ],
    "pages": [2, 3]
  },
    {
      "question_number": 12,
      "ocr_text": " 12. This is a graph of  pressure vs volume. And here is an unreferenced diagram  on this page.<graph_1> and <diagram_2> ",
      "diagrams": [
        {
          "id": "graph_1",
          "coordinates": "0.5,0.5,0.2,0.3",
          "diagram_class": "graph ",
          "page_number": 3
        },
        {
          "id": "diagram_2",
          "coordinates": "0.1,0.1,0.05,0.05",
          "diagram_class": "diagram",
          "page_number": 3
      }
    ],
    "pages": [3, 4]
  },
  ]
}
"""
    # Start processing all PDFs in the input directory
    process_all_pdfs(input_pdf_dir, output_json_dir, prompt)


Processing /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/Physics_pdf_docx_human_ocr/12_1002140198994121111692513661.pdf...
🎉 Output saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/pdf_docx_json/12_1002140198994121111692513661/12_1002140198994121111692513661.json
Processing /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/Physics_pdf_docx_human_ocr/09_1002114885961841111690700733.pdf...
🎉 Output saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/pdf_docx_json/09_1002114885961841111690700733/09_1002114885961841111690700733.json
Processing /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physi

## to clean the json 

In [7]:
import os
import json

def clean_json_content(file_path):
    """
    Cleans the JSON file by removing the first and last lines from the file.
    """
    try:
        # Check if the file is empty
        if os.path.getsize(file_path) == 0:
            print(f"Skipping empty file: {file_path}")
            return

        # Open the file to read the raw content
        with open(file_path, 'r') as file:
            lines = file.readlines()

        # Ensure the file has more than two lines (i.e., has content to remove from both ends)
        if len(lines) <= 2:
            print(f"Skipping file with not enough content to clean: {file_path}")
            return
        
        # Remove the first and last lines
        cleaned_lines = lines[1:-1]

        # Join the cleaned lines and load the cleaned data as JSON
        cleaned_data = "".join(cleaned_lines)
        try:
            data = json.loads(cleaned_data)
        except json.JSONDecodeError as e:
            print(f"Error reading JSON from file {file_path}: {e}")
            return

        # Save the cleaned JSON data back to the file
        with open(file_path, 'w') as file:
            json.dump(data, file, indent=2)
        
        print(f"Successfully cleaned and saved: {file_path}")
    
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")

def clean_json_files_in_directory(directory_path):
    """
    Loops through the directory and all its subdirectories, cleaning each JSON file.
    """
    for subdir, _, files in os.walk(directory_path):
        for filename in files:
            if filename.endswith('.json'):
                file_path = os.path.join(subdir, filename)
                clean_json_content(file_path)

# Set the directory path where the JSON files are located
directory_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/pdf_docx_json"

# Clean all JSON files in the directory and its subdirectories
clean_json_files_in_directory(directory_path)


Successfully cleaned and saved: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/pdf_docx_json/12_1002140198994121111692513661/12_1002140198994121111692513661.json
Successfully cleaned and saved: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/pdf_docx_json/09_1002114885961841111690700733/09_1002114885961841111690700733.json
Successfully cleaned and saved: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/pdf_docx_json/10_10021138351083421111694954514/10_10021138351083421111694954514.json
Successfully cleaned and saved: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/pdf_docx_json/02_1002137635994121111692517447/02_1002137635994121111692517447.json
Successfully clean

## to break the json into each question for each chapter

In [42]:
import os
import json

def create_solution_jsons(input_file_path, output_directory):
    """
    Create individual solution JSON files for each question in the input JSON.
    """
    try:
        # Read the input JSON file
        with open(input_file_path, 'r') as file:
            data = json.load(file)
        
        # Extract the file name without extension and create a new directory for the solution files
        directory_name = os.path.splitext(os.path.basename(input_file_path))[0]
        
        # Extract the first part of the directory name (before the first '_')
        folder_prefix = directory_name.split('_')[0]  # Gets the first part before '_'
        
        # Define output folder
        output_folder = os.path.join(output_directory, directory_name)
        os.makedirs(output_folder, exist_ok=True)
        
        # Loop through each question and create a new JSON for each
        for entry in data:
            question_number = entry.get('question_number')
            solution = [{
                "question_number": question_number,
                "ocr_text": entry.get('ocr_text'),
                "diagrams": entry.get('diagrams', []),
                "pages": entry.get('pages', [])
            }]
            
            # Define the solution file path with the new naming convention
            solution_file_path = os.path.join(output_folder, f"{folder_prefix}_solution_{question_number}.json")
            
            # Write the solution JSON to the file
            with open(solution_file_path, 'w') as solution_file:
                json.dump(solution, solution_file, indent=2)
            
            print(f"Solution {question_number} saved in {solution_file_path}")
    
    except Exception as e:
        print(f"Error processing file {input_file_path}: {e}")

def process_json_files_in_directory(directory_path, output_directory):
    """
    Process all JSON files in the specified directory and its subdirectories.
    """
    for subdir, _, files in os.walk(directory_path):
        for filename in files:
            if filename.endswith('.json'):
                file_path = os.path.join(subdir, filename)
                create_solution_jsons(file_path, output_directory)


# Set the directory path where the JSON files are located
directory_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/pdf_docx_json"  # Replace with the actual path

# Set the output directory where you want to save the solutions
output_directory = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/solutions_chapter"

# Process all JSON files in the directory and its subdirectories
process_json_files_in_directory(directory_path, output_directory)


Solution 1 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/solutions_chapter/12_1002140198994121111692513661/12_solution_1.json
Solution 2 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/solutions_chapter/12_1002140198994121111692513661/12_solution_2.json
Solution 3 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/solutions_chapter/12_1002140198994121111692513661/12_solution_3.json
Solution 4 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/solutions_chapter/12_1002140198994121111692513661/12_solution_4.json
Solution 1 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_

## send the PREDICTION IMAGES TO GEMINI PRO

In [15]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
import fitz  # PyMuPDF for PDF processing
from PIL import Image
import json
import base64
import cv2
import time
import numpy as np

# === Load API Key ===
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))

# Set model
model_name = "gemini-2.5-pro"
model = genai.GenerativeModel(model_name)

# === Prompt for Gemini ===
PROMPT = """
### System Instruction

**Role**: You are a meticulous digital archivist tasked with transcribing handwritten student answer sheets.

**Core Task**: Your goal is to create a perfect digital copy of the student's work.

**Important Rule**: **DO NOT** correct any spelling, punctuation, or grammatical errors. This includes cases where words might seem misspelled, such as "metabolities" instead of "metabolites". **Preserve all text exactly as it appears in the PDF**, even if there are apparent mistakes or inconsistencies.
- Whenever strikethrough or crossed-out text is detected in handwritten content, ensure that the crossed-out text is completely omitted from the transcription. Only include the uncrossed portion of the text, preserving the rest as it is.
- Ensure that question numbers are identified and formatted correctly in the JSON output, maintaining proper sequencing. For example, if the question number is 2, it should appear as '2' in the output. 
- Please avoid using Unicode escape sequences (e.g., \u25b3) and provide direct characters like ∆, ², and cm.
- Whenever a table is found in the image, output the information in a proper table format.
- Please ignore symbols like  "->" from the final text output.
- If the word "Ans." is followed by a number (e.g., "Ans1."), only include the number and exclude the word "Ans."
- Whenever subscripts or superscripts are present in the OCR text, ensure they are formatted accordingly. For example, 'h_i' should be rendered as 'hᵢ' for subscripts and 'x²' should be rendered as 'x²' for superscripts, preserving the correct formatting for both.
---
### Other Directives
1.  **Ignore Page Template**: Exclude all non-content elements like headers( for eg: subject name), footers, page numbers, or decorative logos.
2.  **Collate Sub-Questions**: Group all parts of a question (e.g., 11. (1), 11. (2)) under a single main question number. Preserve the original sub-question numbering in the text.

### Output Format
- The output MUST be a single, valid JSON array containing one object per main question.
- Do NOT include any text or explanations outside of the JSON array.

**Example of a valid JSON object:**
[
 {
    "question_number": 11,
    "ocr_text": "11. (1) This is the answer to the first part. 11. (2) This is the final answer, which contains a PV curve <diagram_1>. [NOTE: The student originally wrote 'the initial answer' here and struck it through; it has been correctly omitted from this output per the critical rule.]<diagram_1>",
    "diagrams": [
      {
        "id": "diagram_1",
        "coordinates": "0.5,0.5,0.2,0.3",
        "diagram_class": "diagram",
        "page_number": 3
      }
    ],
    "pages": [2, 3]
  },
    {
      "question_number": 12,
      "ocr_text": " 12. This is a graph of  pressure vs volume. And here is an unreferenced diagram  on this page.<graph_1> and <diagram_2> ",
      "diagrams": [
        {
          "id": "graph_1",
          "coordinates": "0.5,0.5,0.2,0.3",
          "diagram_class": "graph ",
          "page_number": 3
        },
        {
          "id": "diagram_2",
          "coordinates": "0.1,0.1,0.05,0.05",
          "diagram_class": "diagram",
          "page_number": 3
      }
    ],
    "pages": [3, 4]
  },
]


**Schema Definitions:**
- `question_number` (integer): The main question number.
- `ocr_text` (string): The full, collated text for the question and all its sub-parts.
- `diagrams` (array): A list of diagram objects. Leave as an empty array `[]` if none.
  - `id` (string): The diagram identifier from the text.
  - `coordinates` (string): "x_mid,y_mid,width,height", with values normalized between 0 and 1 relative to image dimensions.
  - `page_number` (integer): The page where the diagram is located.
  - diagram_class: mention is it a diagram or a graph .
- `pages` (array): A list of all page numbers on which any part of the question appears.
"""

# === Set the dimension value ===
dim = 736  # Define the dimension value

# === Resize Image Function ===
def resize_image(image, dim=dim, save_path=None):
    image1 = np.array(image.convert('RGB'))  # Ensure the image is in RGB mode
    original_size = image1.shape  # (height, width, channels)
    image1 = image1.mean(axis=2)  # Convert image to grayscale
    h, w = image1.shape
    if w > h:
        new_w = dim
        new_h = int(h * (dim / w))
    else:
        new_h = dim
        new_w = int(w * (dim / h))
    resized_image = cv2.resize(image1, (new_w, new_h), interpolation=cv2.INTER_AREA)
    resized_image_pil = Image.fromarray(resized_image)
    resized_image_pil = resized_image_pil.convert('RGB')  # Convert to RGB before saving
    if save_path:
        resized_image_pil.save(save_path)
    return original_size, (new_h, new_w), resized_image_pil

# === Load PDF, Convert Pages to Images ===
def pdf_to_images(pdf_path, output_folder):
    doc = fitz.open(pdf_path)
    images = []
    num_pages = doc.page_count
    for i in range(num_pages):
        page = doc.load_page(i)
        pix = page.get_pixmap(dpi=300)
        img_path_default = os.path.join(output_folder, f"page_{i + 1}.jpeg")
        pix.save(img_path_default)
        images.append(img_path_default)
    return images, num_pages

# === Load Base64 Images ===
def load_base64_images(folder_path, num_pages):
    b64_list = []
    for i in range(num_pages):
        img_filename_dim = f"DIM_{dim}_PAGE_{i + 1}.jpeg"  # Use dim variable here
        img_filename_default = f"page_{i + 1}.jpeg"  # Old naming convention

        img_path_dim = os.path.join(folder_path, img_filename_dim)
        img_path_default = os.path.join(folder_path, img_filename_default)

        if os.path.exists(img_path_dim):
            path = img_path_dim
        elif os.path.exists(img_path_default):
            path = img_path_default
        else:
            print(f"❌ Image {img_filename_dim} or {img_filename_default} not found in {folder_path}")
            continue

        with open(path, "rb") as f:
            b64_list.append(base64.b64encode(f.read()).decode())
    return b64_list

# === Batch send to Gemini ===
def send_to_gemini(resized_images_objs):
    try:
        response = model.generate_content([PROMPT] + resized_images_objs)  # Batch processing
        raw = response.text.strip()
        cleaned = raw.strip('```json').strip('```').strip()
        parsed = json.loads(cleaned)
        return parsed
    except Exception as e:
        print(f"❌ Failed to process images: {e}")
        return None

# === Function to process all PDF files in a directory ===
def process_all_pdfs(input_pdf_dir, output_json_dir):
    # Iterate through all files in the directory
    for root, dirs, files in os.walk(input_pdf_dir):
        for file in files:
            if file.endswith(".pdf"):
                input_pdf_path = os.path.join(root, file)
                print(f"Processing {input_pdf_path}...")  # Log the current file being processed
                # Define the output folder path using the PDF filename
                pdf_filename = os.path.splitext(file)[0]
                output_folder = os.path.join(output_json_dir, pdf_filename)
                os.makedirs(output_folder, exist_ok=True)

                # Process each PDF file
                try:
                    # Step 1: Convert PDF to images
                    images, num_pages = pdf_to_images(input_pdf_path, output_folder)

                    # Step 2: Resize images
                    resized_images = []
                    for page_num in range(len(images)):
                        img_path = images[page_num]
                        original_size, new_size, resized_image = resize_image(Image.open(img_path), dim=dim)
                        
                        # Save resized image
                        img_filename_dim = f"DIM_{dim}_PAGE_{page_num + 1}.jpeg"
                        resized_image.save(os.path.join(output_folder, img_filename_dim))
                        
                        resized_images.append(resized_image)

                    # Step 3: Load base64 images
                    images_b64 = load_base64_images(output_folder, len(images))

                    # Step 4: Send to Gemini for OCR
                    results = send_to_gemini(resized_images)

                    if results:
                        output_json_filename = f"{pdf_filename}.json"
                        json_path = os.path.join(output_folder, output_json_filename)

                        # Save the OCR results
                        with open(json_path, "w") as f:
                            json.dump(results, f, indent=3)

                        print(f"OCR results saved to {json_path}")
                    else:
                        print("❌ No results from Gemini OCR")

                except Exception as e:
                    print(f"❌ Error processing {input_pdf_path}: {e}")


# === Example Usage ===
if __name__ == "__main__":
    input_pdf_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy"  # Change this to the correct directory path
    output_json_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy_json"  # Change this to the desired output directory
    
    # Process all PDFs in the input directory
    process_all_pdfs(input_pdf_dir, output_json_dir)


Processing /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy/12_1002140198994121111692513661.pdf...
OCR results saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy_json/12_1002140198994121111692513661/12_1002140198994121111692513661.json
Processing /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy/09_1002114885961841111690700733.pdf...
OCR results saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy_json/09_1002114885961841111690700733/09_1002114885961841111690700733.json
Processing /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy/04_100210408219741901111

## to break the json into each quesiton 

In [41]:
import os
import json

def create_solution_jsons(input_file_path, output_directory):
    """
    Create individual solution JSON files for each question in the input JSON.
    """
    try:
        # Read the input JSON file
        with open(input_file_path, 'r') as file:
            data = json.load(file)
        
        # Extract the file name without extension and create a new directory for the solution files
        directory_name = os.path.splitext(os.path.basename(input_file_path))[0]
        
        # Extract the first part of the directory name (before the first '_')
        folder_prefix = directory_name.split('_')[0]  # Gets the first part before '_'
        
        # Define output folder
        output_folder = os.path.join(output_directory, directory_name)
        os.makedirs(output_folder, exist_ok=True)
        
        # Loop through each question and create a new JSON for each
        for entry in data:
            question_number = entry.get('question_number')
            solution = [{
                "question_number": question_number,
                "ocr_text": entry.get('ocr_text'),
                "diagrams": entry.get('diagrams', []),
                "pages": entry.get('pages', [])
            }]
            
            # Define the solution file path with the new naming convention
            solution_file_path = os.path.join(output_folder, f"{folder_prefix}_solution_{question_number}.json")
            
            # Write the solution JSON to the file
            with open(solution_file_path, 'w') as solution_file:
                json.dump(solution, solution_file, indent=2)
            
            print(f"Solution {question_number} saved in {solution_file_path}")
    
    except Exception as e:
        print(f"Error processing file {input_file_path}: {e}")

def process_json_files_in_directory(directory_path, output_directory):
    """
    Process all JSON files in the specified directory and its subdirectories.
    """
    for subdir, _, files in os.walk(directory_path):
        for filename in files:
            if filename.endswith('.json'):
                file_path = os.path.join(subdir, filename)
                create_solution_jsons(file_path, output_directory)

# Set the directory path where the JSON files are located
directory_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy_json"  # Replace with the actual path

# Set the output directory where you want to save the solutions
output_directory = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/solutions"

# Process all JSON files in the directory and its subdirectories
process_json_files_in_directory(directory_path, output_directory)


Solution 1 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/solutions/12_1002140198994121111692513661/12_solution_1.json
Solution 2 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/solutions/12_1002140198994121111692513661/12_solution_2.json
Solution 3 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/solutions/12_1002140198994121111692513661/12_solution_3.json
Solution 4 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/solutions/12_1002140198994121111692513661/12_solution_4.json
Solution 1 saved in /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/s

# to make a table of human and gemini ocr

In [16]:
import os
import json

def compare_ocr_folders(human_ocr_path, gemini_ocr_path, solution_folder, solution_number, file_prefix):
    human_solution_path = os.path.join(
        human_ocr_path, 'Physics/Physics_human/solutions_chapter', solution_folder,
        f"{file_prefix}_solution_{solution_number}.json"
    )
    gemini_solution_path = os.path.join(
        gemini_ocr_path, 'Physics/Physics_Gemini/solutions', solution_folder,
        f"{file_prefix}_solution_{solution_number}.json"
    )
    
    print(f"Processing: {human_solution_path}")
    print(f"Processing: {gemini_solution_path}")
    
    md_content = "| Human_OCR | Gemini_OCR |\n"
    md_content += "|-----------|------------|\n"
    
    human_content = "NA"
    gemini_content = "NA"
    
    # Human OCR
    if os.path.exists(human_solution_path):
        with open(human_solution_path, 'r') as human_file:
            try:
                human_data = json.load(human_file)
                if isinstance(human_data, list) and len(human_data) > 0:
                    human_content = human_data[0].get('ocr_text', 'NA')
                    print(f"Human OCR ocr_text: {repr(human_content)}")
                else:
                    print(f"Warning: {human_solution_path} is not a list or is empty.")
            except Exception as e:
                print(f"Error reading {human_solution_path}: {e}")
    else:
        print(f"File not found: {human_solution_path}")
    
    # Gemini OCR
    if os.path.exists(gemini_solution_path):
        with open(gemini_solution_path, 'r') as gemini_file:
            try:
                gemini_data = json.load(gemini_file)
                if isinstance(gemini_data, list) and len(gemini_data) > 0:
                    gemini_content = gemini_data[0].get('ocr_text', 'NA')
                    print(f"Gemini OCR ocr_text: {repr(gemini_content)}")
                else:
                    print(f"Warning: {gemini_solution_path} is not a list or is empty.")
            except Exception as e:
                print(f"Error reading {gemini_solution_path}: {e}")
    else:
        print(f"File not found: {gemini_solution_path}")
    
    # Replace newlines for markdown cell
    human_content = human_content.replace("\n", "<br>") if isinstance(human_content, str) else "NA"
    gemini_content = gemini_content.replace("\n", "<br>") if isinstance(gemini_content, str) else "NA"
    
    md_content += f"| {human_content} | {gemini_content} |\n"
    
    output_dir = f'/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables/{solution_folder}'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    md_filename = os.path.join(output_dir, f"{file_prefix}_solution_{solution_number}_table.md")
    with open(md_filename, 'w') as md_file:
        md_file.write(md_content)
    print(f"Created {md_filename}")

def process_folders(human_ocr_base_path, gemini_ocr_base_path):
    human_ocr_path = os.path.join(human_ocr_base_path, 'Physics/Physics_human/solutions_chapter')
    gemini_ocr_path = os.path.join(gemini_ocr_base_path, 'Physics/Physics_Gemini/solutions')
    
    for solution_folder in os.listdir(human_ocr_path):
        solution_folder_path_human = os.path.join(human_ocr_path, solution_folder)
        if not os.path.isdir(solution_folder_path_human):
            continue
        for fname in os.listdir(solution_folder_path_human):
            if fname.endswith('.json') and '_solution_' in fname:
                try:
                    file_prefix = fname.split('_solution_')[0]
                    solution_number = int(fname.split('_solution_')[1].split('.')[0])
                except Exception:
                    print(f"Filename parse error: {fname}")
                    continue
                compare_ocr_folders(human_ocr_base_path, gemini_ocr_base_path, solution_folder, solution_number, file_prefix)

# Set paths to your directories
human_ocr_base_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr'
gemini_ocr_base_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr'

# Process the folders
process_folders(human_ocr_base_path, gemini_ocr_base_path)

Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/solutions_chapter/12_1002140198994121111692513661/12_solution_1.json
Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/solutions/12_1002140198994121111692513661/12_solution_1.json
Human OCR ocr_text: '1. Convex lens'
Gemini OCR ocr_text: '1) Convex lens'
Created /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables/12_1002140198994121111692513661/12_solution_1_table.md
Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_human/solutions_chapter/12_1002140198994121111692513661/12_solution_2.json
Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/

## to make cer

In [17]:
import os
import json
from difflib import ndiff

def char_error_rate(s1, s2):
    """
    Calculate the character error rate (CER) between two strings.
    Returns 'na' if either string is 'na'.
    """
    if s1 == "na" or s2 == "na":
        return "na"
    diff = list(ndiff(s1, s2))
    insertions = sum(1 for d in diff if d[0] == '+')
    deletions = sum(1 for d in diff if d[0] == '-')
    ref_len = len(s1)
    if ref_len == 0:
        return 0 if len(s2) == 0 else 1
    cer = (insertions + deletions) / ref_len
    return cer

def highlight_differences(s1, s2):
    """
    Highlight the differences between two strings.
    Returns 'na' if either string is 'na'.
    """
    if s1 == "na" or s2 == "na":
        return "na"
    diff = list(ndiff(s1, s2))
    result = []
    for d in diff:
        if d[0] == ' ':
            result.append(d[2])
        elif d[0] == '-':
            result.append(f"[-{d[2]}-]")
        elif d[0] == '+':
            result.append(f"[+{d[2]}+]")
    return ''.join(result)

def compare_ocr_folders(human_ocr_path, gemini_ocr_path, solution_folder, solution_number):
    # Extract the first part of the folder name (before the first '_') as the prefix
    folder_prefix = solution_folder.split('_')[0]
    
    # Construct the paths to the solution files for Human OCR and Gemini OCR using the folder prefix
    human_solution_path = os.path.join(human_ocr_path, 'Physics/Physics_human/solutions_chapter', solution_folder, f"{folder_prefix}_solution_{solution_number}.json")
    gemini_solution_path = os.path.join(gemini_ocr_path, 'Physics/Physics_Gemini/solutions', solution_folder, f"{folder_prefix}_solution_{solution_number}.json")
    
    # Prepare the markdown table header
    md_content = "| Human_OCR               | Gemini_OCR              | cer   | highlight difference   |\n"
    md_content += "|-------------------------|-------------------------|-------|------------------------|\n"
    
    # Initialize content variables
    human_content = "NA"
    gemini_content = "NA"
    cer = "NA"
    highlight_diff = "NA"
    
    # Check if the Human OCR solution file exists and extract the 'ocr_text'
    if os.path.exists(human_solution_path):
        with open(human_solution_path, 'r') as human_file:
            human_data = json.load(human_file)
        
        if isinstance(human_data, list) and len(human_data) > 0:
            human_content = human_data[0].get('ocr_text', 'NA')
    
    # Check if the Gemini OCR solution file exists and extract the 'ocr_text'
    if os.path.exists(gemini_solution_path):
        with open(gemini_solution_path, 'r') as gemini_file:
            gemini_data = json.load(gemini_file)
        
        if isinstance(gemini_data, list) and len(gemini_data) > 0:
            gemini_content = gemini_data[0].get('ocr_text', 'NA')
    
    # Replace line breaks with <br> in both OCR contents (even if it's "NA")
    human_content = human_content.replace("\n", "<br>")
    gemini_content = gemini_content.replace("\n", "<br>")
    
    # If either Human OCR or Gemini OCR is "NA", set CER and highlight differences to "NA"
    if human_content == "NA" or gemini_content == "NA":
        cer = "NA"
        highlight_diff = "NA"
    else:
        # Calculate CER and highlight differences
        cer = char_error_rate(human_content, gemini_content)
        highlight_diff = highlight_differences(human_content, gemini_content)
    
    # Add the extracted content to the table as a single row per solution
    md_content += f"| {human_content} | {gemini_content} | {cer} | {highlight_diff} |\n"
    
    # Save the markdown file to the specific solution folder with the dynamic filename
    output_dir = f'/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables_cer/{solution_folder}'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Generate the markdown file for the solution with the correct filename including the prefix
    md_filename = os.path.join(output_dir, f"{folder_prefix}_solution_{solution_number}_table.md")
    with open(md_filename, 'w') as md_file:
        md_file.write(md_content)
    print(f"Created {md_filename}")

def process_folders(human_ocr_base_path, gemini_ocr_base_path):
    human_ocr_path = os.path.join(human_ocr_base_path, 'Physics/Physics_human/solutions_chapter')
    gemini_ocr_path = os.path.join(gemini_ocr_base_path, 'Physics/Physics_Gemini/solutions')
    
    for solution_folder in os.listdir(human_ocr_path):
        solution_folder_path_human = os.path.join(human_ocr_path, solution_folder)
        if not os.path.isdir(solution_folder_path_human):
            continue
        for fname in os.listdir(solution_folder_path_human):
            if fname.endswith('.json') and '_solution_' in fname:
                try:
                    file_prefix = fname.split('_solution_')[0]
                    solution_number = int(fname.split('_solution_')[1].split('.')[0])
                except Exception:
                    print(f"Filename parse error: {fname}")
                    continue
                compare_ocr_folders(human_ocr_base_path, gemini_ocr_base_path, solution_folder, solution_number)

# Set paths to your directories
human_ocr_base_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr'
gemini_ocr_base_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr'

# Process the folders
process_folders(human_ocr_base_path, gemini_ocr_base_path)



Created /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables_cer/12_1002140198994121111692513661/12_solution_1_table.md
Created /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables_cer/12_1002140198994121111692513661/12_solution_2_table.md
Created /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables_cer/12_1002140198994121111692513661/12_solution_3_table.md
Created /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables_cer/12_1002140198994121111692513661/12_solution_4_table.md
Created /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables_cer/09_1002114885961841111690700733/09_solution_4_table.md
Created /Users/simrannaik/Desktop/solution_improve

# to send gemini 

In [19]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

# === Load API Key ===
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))

# Set model
model = genai.GenerativeModel("gemini-2.5-pro")

def send_md_and_prompt(input_md_path, prompt, output_md_dir):
    # Read the Markdown file
    with open(input_md_path, 'r', encoding='utf-8') as f:
        md_content = f.read()

    # Compose the prompt
    full_prompt = f"{prompt}\n\n<Markdown Table Input>\n{md_content}"

    try:
        # Generate content with the AI model
        response = model.generate_content(
            full_prompt,
            generation_config={"temperature": 0.2},
        )
        generated_text = response.text

        # Try to extract Markdown from the response
        if generated_text.strip().startswith('```markdown'):
            generated_text = generated_text.strip().removeprefix('```markdown').removesuffix('```').strip()
        elif generated_text.strip().startswith('```'):
            generated_text = generated_text.strip().removeprefix('```').removesuffix('```').strip()

        # Extract prefix from the parent folder name
        solution_folder_name = os.path.basename(os.path.dirname(input_md_path))
        prefix = solution_folder_name.split('_')[0]  # Extract prefix part (e.g., "12")

        # Extract solution number from the file name
        base_name = os.path.splitext(os.path.basename(input_md_path))[0]
        
        # Extract the solution number correctly from filenames like '12_solution_1_table.md'
        # This assumes that the solution number comes after 'solution_' in the filename
        solution_number = base_name.split('_')[2]  # Correctly extract the solution number part

        # Create the output folder and file path
        output_solution_dir = os.path.join(output_md_dir, solution_folder_name)
        os.makedirs(output_solution_dir, exist_ok=True)
        
        # Construct output file path using prefix and solution number
        output_md_path = os.path.join(output_solution_dir, f"{prefix}_solution_{solution_number}_table.md")

        # Save the generated output
        with open(output_md_path, 'w', encoding='utf-8') as out_file:
            out_file.write(generated_text)

        print(f"🎉 Output saved to {output_md_path}")

    except Exception as e:
        print(f"❌ Error generating response: {e}")

# Function to process all .md files in a directory
def process_all_md_files(input_dir, output_dir, prompt):
    # Walk through all files in the directory
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".md"):
                input_md_path = os.path.join(root, file)
                output_md_dir = output_dir
                print(f"Processing {input_md_path}...")  # Log the current file being processed
                send_md_and_prompt(input_md_path, prompt, output_md_dir)

# === Example Usage ===
if __name__ == "__main__":
    input_md_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables"  # Directory containing .md files
    output_md_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/table_analysis"  # Directory for saving processed files
    prompt = """
Role: You are a highly accurate and detail-oriented Quality Assurance (QA) Engine designed to evaluate the performance of OCR systems.

Context: You will receive a Markdown table containing two columns:
Human OCR: Text extracted manually by a human. This is treated as the reference or ground truth, though it may contain minor errors.
Gemini OCR: The text extracted by the Gemini model, which may also contain errors.
Objective: Your goal is to compare each row in the Human OCR and Gemini OCR columns and identify any discrepancies only if the Gemini OCR does not match the Human OCR exactly. The Human OCR is the reference, and Gemini OCR should align with it, including any errors that may exist in Human OCR. Please follow the guidelines below:

### Special Cases:
1. **Flag as "No Errors"**: If both versions are identical and correct, flag the Discrepancy Analysis as "NO ERRORS" and the Type of Error as "NO ERRORS".
2. "If no discrepancies are found between the Human OCR Text and Gemini OCR Text, except for 'NA', flag the 'Discrepancy Analysis' as 'NO ERRORS' and 'Type of Error' as 'NO ERRORS'."
3. **Type of Error**: Choose from Spelling, Wording, Extra Content, Punctuation, Numerical Difference, Missing Content, Content Mix-up, Omission.
4. even if the ocr has multiple errors name all the errors in the type of errors columnn.


### Decision-Making Guidelines:
1.Human OCR is Reference: Assume Human OCR is correct unless clearly erroneous. Any mistake present in the Human OCR must also appear in Gemini OCR to be acceptable.

2.Spelling Errors: If Human OCR contains a spelling mistake and Gemini OCR silently corrects it, flag it. Gemini OCR must replicate the same spelling.

3.Wording Differences: Any difference in phrasing, sentence structure, or choice of words that deviates from Human OCR should be flagged.

4.Punctuation: Gemini OCR must match the punctuation used in Human OCR. If punctuation differs and it affects readability or meaning, flag it.

5.Numerical Discrepancies: All numbers (e.g., page numbers, values, equations) must match. If Gemini OCR shows different numerals, flag it.

6.Missing or Extra Content: If Gemini OCR omits any content present in Human OCR or adds extra text, flag it.

7.Content Mix-Up: If text from different sections or rows appears to be jumbled or misaligned in Gemini OCR, flag it.

8.Omission or Placeholder Values: If either version contains "NA" or is missing data while the other has content, flag it as an omission. 

### Reporting:
For each row, if there is a discrepancy, report the following in a Markdown table:

| Human OCR               | Gemini OCR              | Type of Error  | Discrepancy Analysis   |
|-------------------------|-------------------------|----------------|------------------------|
| [Human OCR Text]        | [Gemini OCR Text]       | [Error Type]   | [Error Analysis]       |

"""
    # Call the function to process all markdown files in the directory
    process_all_md_files(input_md_dir, output_md_dir, prompt)


Processing /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables/12_1002140198994121111692513661/12_solution_1_table.md...
🎉 Output saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/table_analysis/12_1002140198994121111692513661/12_solution_1_table.md
Processing /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables/12_1002140198994121111692513661/12_solution_3_table.md...
🎉 Output saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/table_analysis/12_1002140198994121111692513661/12_solution_3_table.md
Processing /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/tables/12_1002140198994121111692513661/12_solution_2_table.md...
🎉 Output saved t

## merge the cer and gemini tables

In [20]:
import os

base_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics"
analysis_dir = os.path.join(base_dir, "table_analysis")
cer_dir = os.path.join(base_dir, "tables_cer")
final_dir = os.path.join(base_dir, "final_tables")

def parse_md_table(md_path):
    rows = []
    with open(md_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or set(line.replace('|', '').replace(':', '').replace('-', '')) == set():
                continue
            if line.startswith('|'):
                parts = [cell.strip() for cell in line.strip('|').split('|')]
                rows.append(parts)
    return rows

def write_md_table(md_path, rows):
    with open(md_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(rows):
            f.write("| " + " | ".join(row) + " |\n")
            if i == 0:
                f.write("|" + "|".join(['---'] * len(row)) + "|\n")

# Loop over all subfolders in table_analysis
for folder_id in os.listdir(analysis_dir):
    analysis_path = os.path.join(analysis_dir, folder_id)
    cer_path = os.path.join(cer_dir, folder_id)
    out_path = os.path.join(final_dir, folder_id)
    if not os.path.isdir(analysis_path) or not os.path.isdir(cer_path):
        continue
    os.makedirs(out_path, exist_ok=True)

    for fname in os.listdir(analysis_path):
        if not fname.endswith(".md"):
            continue
        analysis_file = os.path.join(analysis_path, fname)
        cer_file = os.path.join(cer_path, fname)
        if not os.path.exists(cer_file):
            print(f"Skipping {fname} in {folder_id}: no matching file in tables_cer")
            continue

        analysis_rows = parse_md_table(analysis_file)
        cer_rows = parse_md_table(cer_file)

        if len(analysis_rows) < 2 or len(analysis_rows[0]) < 2:
            print(f"Skipping {fname} in {folder_id}: not enough columns/rows in analysis table")
            continue
        last2_header = analysis_rows[0][-2:]
        last2_row = analysis_rows[1][-2:]

        new_rows = []
        for i, row in enumerate(cer_rows):
            if i == 0:
                new_rows.append(row + last2_header)
            elif i == 1:
                new_rows.append(row + last2_row)
            else:
                new_rows.append(row)

        out_fname = fname.replace("solution", "final")
        out_file = os.path.join(out_path, out_fname)
        write_md_table(out_file, new_rows)
        print(f"Saved combined table: {out_file}")


Saved combined table: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/final_tables/12_1002140198994121111692513661/12_final_1_table.md
Saved combined table: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/final_tables/12_1002140198994121111692513661/12_final_3_table.md
Saved combined table: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/final_tables/12_1002140198994121111692513661/12_final_2_table.md
Saved combined table: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/final_tables/12_1002140198994121111692513661/12_final_4_table.md
Saved combined table: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/final_tables/09_1002114885961841111690700733/09_fi

## make a final table 

In [21]:
import os

base_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics"
final_tables_dir = os.path.join(base_dir, "final_tables")
output_file = os.path.join(base_dir, "final_table.md")

all_rows = []
header = None

def parse_md_table(md_path):
    rows = []
    with open(md_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            # Skip empty lines and separator lines
            if not line or set(line.replace('|', '').replace(':', '').replace('-', '')) == set():
                continue
            if line.startswith('|'):
                parts = [cell.strip() for cell in line.strip('|').split('|')]
                rows.append(parts)
    return rows

for folder_id in os.listdir(final_tables_dir):
    folder_path = os.path.join(final_tables_dir, folder_id)
    if not os.path.isdir(folder_path):
        continue
    for fname in os.listdir(folder_path):
        if not fname.endswith(".md"):
            continue
        file_path = os.path.join(folder_path, fname)
        rows = parse_md_table(file_path)
        if not rows or len(rows) < 2:
            print(f"Warning: {file_path} does not have enough rows.")
            continue
        if header is None:
            header = ["FILE_NAME"] + rows[0]
        # Find the first data row (skip header and separator)
        for row in rows[1:]:
            # Skip separator rows (all cells are --- or empty)
            if all(cell.strip('- ') == '' for cell in row):
                continue
            all_rows.append([fname] + row)
            break
        else:
            print(f"Warning: {file_path} has no data row.")

# Write merged table
with open(output_file, "w", encoding="utf-8") as f:
    if header:
        f.write("| " + " | ".join(header) + " |\n")
        f.write("|" + "|".join(['---'] * len(header)) + "|\n")
    for row in all_rows:
        f.write("| " + " | ".join(row) + " |\n")

print(f"Merged data rows written to {output_file}")

Merged data rows written to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/final_table.md


## make a average cer and no of errors analysis

In [22]:
import os

input_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/final_table.md"

cer_values = []
na_count = 0
row_count = 0
cer_col_idx = None

with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Find header and cer column index
for i, line in enumerate(lines):
    if line.strip().startswith("|") and "cer" in line:
        header = [cell.strip() for cell in line.strip().strip('|').split('|')]
        try:
            cer_col_idx = header.index("cer")
        except ValueError:
            raise Exception("No 'cer' column found in header!")
        break

if cer_col_idx is None:
    raise Exception("No header with 'cer' column found!")

# Process data rows
for line in lines[i+2:]:  # skip header and separator
    if not line.strip().startswith("|"):
        continue
    cells = [cell.strip() for cell in line.strip().strip('|').split('|')]
    if len(cells) <= cer_col_idx:
        continue
    cer_val = cells[cer_col_idx]
    if cer_val.lower() == "na":
        na_count += 1
    else:
        try:
            cer_values.append(float(cer_val))
        except ValueError:
            continue
    row_count += 1

# Calculate average
average_cer = sum(cer_values) / len(cer_values) if cer_values else 0

print(f"Total data rows: {row_count}")
print(f"Total 'na' in cer column: {na_count}")
print(f"Average cer (excluding 'na'): {average_cer}")

Total data rows: 109
Total 'na' in cer column: 11
Average cer (excluding 'na'): 0.44544196395002966


In [24]:
import os
from collections import Counter

input_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/final_table.md"

# List of error types to count
error_types = [
    "Spelling",
    "Wording",
    "Extra Content",
    "Punctuation",
    "Numerical Difference",
    "Missing Content",
    "Content Mix-up",
    "Omission"
]

type_of_error_col_idx = None
gemini_ocr_col_idx = None
human_ocr_col_idx = None
counts = Counter()

with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Find header and column indices
for i, line in enumerate(lines):
    if line.strip().startswith("|") and "Type of Error" in line:
        header = [cell.strip() for cell in line.strip().strip('|').split('|')]
        try:
            type_of_error_col_idx = header.index("Type of Error")
            gemini_ocr_col_idx = header.index("Gemini_OCR")
            human_ocr_col_idx = header.index("Human_OCR")
        except ValueError:
            raise Exception("One or more required columns not found in header!")
        break

if type_of_error_col_idx is None or gemini_ocr_col_idx is None or human_ocr_col_idx is None:
    raise Exception("Necessary columns ('Type of Error', 'Gemini_OCR', 'Human_OCR') not found in header!")

# Process data rows
for line in lines[i+2:]:  # skip header and separator
    if not line.strip().startswith("|"):
        continue
    cells = [cell.strip() for cell in line.strip().strip('|').split('|')]
    if len(cells) <= max(type_of_error_col_idx, gemini_ocr_col_idx, human_ocr_col_idx):
        continue
    error_val = cells[type_of_error_col_idx]
    gemini_ocr = cells[gemini_ocr_col_idx]
    human_ocr = cells[human_ocr_col_idx]

    # Check if either gemini_ocr or human_ocr is "NA"
    if gemini_ocr.lower() == "na" or human_ocr.lower() == "na":
        continue  # Skip this row if either OCR value is "NA"
    
    # Count the error types only if the error type is explicitly mentioned
    if error_val == "NO ERRORS":
        counts[error_val] += 1
    elif error_val in error_types:
        counts[error_val] += 1

# Print results
for error_type in error_types + ["NO ERRORS"]:
    print(f"{error_type}: {counts[error_type]}")


Spelling: 2
Wording: 4
Extra Content: 1
Punctuation: 7
Numerical Difference: 0
Missing Content: 4
Content Mix-up: 1
Omission: 0
NO ERRORS: 9


In [25]:
import os

input_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/final_table.md"

prefix_counts = {f"{i:02d}_final": 0 for i in range(1, 16)}
file_name_col_idx = 0

with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Find header to confirm FILE_NAME is the first column
for i, line in enumerate(lines):
    if line.strip().startswith("|") and "FILE_NAME" in line:
        header = [cell.strip() for cell in line.strip().strip('|').split('|')]
        if header[0] != "FILE_NAME":
            raise Exception("First column is not FILE_NAME!")
        break

# Process data rows
for line in lines[i+2:]:  # skip header and separator
    if not line.strip().startswith("|"):
        continue
    cells = [cell.strip() for cell in line.strip().strip('|').split('|')]
    if len(cells) == 0:
        continue
    file_name = cells[file_name_col_idx]
    for prefix in prefix_counts:
        if file_name.startswith(prefix):
            prefix_counts[prefix] += 1
            break

# Print results
for prefix in prefix_counts:
    print(f"{prefix}: {prefix_counts[prefix]}")

01_final: 11
02_final: 4
03_final: 12
04_final: 6
05_final: 4
06_final: 6
07_final: 12
08_final: 3
09_final: 4
10_final: 12
11_final: 6
12_final: 4
13_final: 11
14_final: 4
15_final: 12
